In [2]:
import arcpy
import os
arcpy.env.overwriteOutput        = True
arcpy.env.outputCoordinateSystem = None
arcpy.CheckOutExtension("Spatial");

In [4]:
target_sr = arcpy.SpatialReference(102003) # target coordinate system: NAD 1983 Contiguous USA Albers — WKID 102003
#target_sr.name
#target_sr.factoryCode

In [5]:
project_folder = r"C:/Users/Chels/OneDrive - University of Illinois - Urbana/Dissertation/Ch4_Sackett_Impacts/Wetland Mapping"
wetland_gdb = os.path.normpath(os.path.join(project_folder, "Illinois_Wetland_Shapefiles.gdb"))

In [6]:
years = [2023, 2024]
n_y = len(years)

# Isolate cropped and medium-to-high intensity polygon inclusions

In [7]:
# define input layers
step6 = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step6_NonDeep_{y}NLCD_Intersect")) for y in years]

In [10]:
# define output layers
step7 = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step7_NonDeep_{y}NLCD_AgDeveloped")) for y in years]

In [9]:
# collect polygon inclusions
for i in range(n_y):
    # export only row crops (82), medium intensity development (23), and high intensity development (24)
    arcpy.conversion.ExportFeatures(
        in_features = step6[i],
        out_features = step7[i],
        where_clause = "gridcode IN (23, 24, 82)"
    )

    # check selection count
    count6 = int(arcpy.management.GetCount(step6[i])[0])
    count7 = int(arcpy.management.GetCount(step7[i])[0])
    print(f"Previous number of features: {count6}")
    print(f"Selected number of features: {count7}")

Previous number of features: 1038049
Selected number of features: 190620
Previous number of features: 1043352
Selected number of features: 190999


# Dissolve to original NWI polygons

In [11]:
dissolve_fields = [
    "NWI_ID",
    "ATTRIBUTE",
    "WETLAND_TY",
    "ACRES",
    "SYSTEM_NAM",
    "SYSTEM_DEF",
    "SUBSYSTEM_",
    "SUBSYSTEM1",
    "CLASS_NAME",
    "CLASS_DEFI",
    "SUBCLASS_N",
    "SUBCLASS_D",
    "SPLIT_CLAS",
    "SPLIT_CL_1",
    "SPLIT_SUBC",
    "SPLIT_SU_1",
    "WATER_REGI",
    "WATER_RE_1",
    "WATER_RE_2",
    "FIRST_MODI",
    "FIRST_MO_1",
    "FIRST_MO_2",
    "FIRST_MO_3",
    "SECOND_MOD",
    "SECOND_M_1",
    "SECOND_M_2",
    "gridcode"
]

In [12]:
# dissolved layer names
step8 = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step8_NonDeep_{y}NLCD_Dissolve_AgDeveloped")) for y in years]

In [13]:
# dissolve polygons to original NWI IDs
for i in range(n_y):
    existing_fields = [f.name for f in arcpy.ListFields(step7[i])]

    # shorten wetland type field name
    if 'WETLAND_TY' not in existing_fields:
        arcpy.management.AlterField(
            in_table=step7[i],
            field='WETLAND_TYPE',
            new_field_name='WETLAND_TY'
        );

    # dissolve
    arcpy.analysis.PairwiseDissolve(
        in_features=step7[i],
        out_feature_class=step8[i],
        dissolve_field=dissolve_fields,
        multi_part="MULTI_PART"
    );

# First modifier: remove farmed and excavated wetlands

In [14]:
# define new output layers
step9 = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step9_NonDeep_{y}NLCD_FirstModifier_AgDeveloped")) for y in years]

In [15]:
for i in range(n_y):
    # export non-farmed and non-excavated
    arcpy.conversion.ExportFeatures(
        in_features = step8[i],
        out_features = step9[i],
        where_clause = "FIRST_MODI IS NULL OR (FIRST_MODI NOT IN ('Farmed','Excavated'))"
    )

    # check selection count
    count8 = int(arcpy.management.GetCount(step8[i])[0])
    count9 = int(arcpy.management.GetCount(step9[i])[0])
    print(f"Previous number of features: {count8}")
    print(f"Selected number of features: {count9}")

Previous number of features: 153870
Selected number of features: 116094
Previous number of features: 153417
Selected number of features: 115670


# Water regime: remove artificially flooded wetlands

In [16]:
# define new output layers
step10 = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_NonDeep_Step10_{y}NLCD_WaterRegime_AgDeveloped")) for y in years]

In [17]:
for i in range(n_y):
    # export non-artificially flooed
    arcpy.conversion.ExportFeatures(
        in_features = step9[i],
        out_features = step10[i],
        where_clause="WATER_REGI IS NULL OR WATER_REGI NOT IN ('Artificially Flooded')"
    );
    
    # check selection count
    count9 = int(arcpy.management.GetCount(step9[i])[0])
    count10 = int(arcpy.management.GetCount(step10[i])[0])
    print(f"Previous number of features: {count9}")
    print(f"Selected number of features: {count10}")

Previous number of features: 116094
Selected number of features: 115685
Previous number of features: 115670
Selected number of features: 115261


# Intersect with counties

In [18]:
census_gdb = os.path.normpath(os.path.join(project_folder, "TIGER_Line_Census.gdb"))

In [19]:
county_layer = os.path.normpath(os.path.join(census_gdb, "tl_2025_il_county_unclipped_NAD1983_Albers"))

In [20]:
county_layer_project = os.path.normpath(os.path.join(census_gdb, "tl_2025_il_county_unclipped_Albers_EqArea"))

In [21]:
arcpy.management.Project(county_layer, county_layer_project, target_sr);

In [22]:
county_spatial_ref = arcpy.Describe(county_layer_project).spatialReference
print(f"Coordinate System: {county_spatial_ref.name}")
print(f"WKID (EPSG): {county_spatial_ref.factoryCode}")

Coordinate System: USA_Contiguous_Albers_Equal_Area_Conic
WKID (EPSG): 102003


In [23]:
step11 = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step11_NonDeep_{y}NLCD_CntyIntersect_AgDeveloped")) for y in years]

In [24]:
for i in range(n_y):
    lyr_spatial_ref = arcpy.Describe(step10[i]).spatialReference
    print(f"Coordinate System: {lyr_spatial_ref.name}")
    print(f"WKID (EPSG): {lyr_spatial_ref.factoryCode}")

    arcpy.analysis.PairwiseIntersect(
        in_features = [step10[i], 
                       county_layer_project],
        out_feature_class = step11[i]
    );

Coordinate System: USA_Contiguous_Albers_Equal_Area_Conic
WKID (EPSG): 102003
Coordinate System: USA_Contiguous_Albers_Equal_Area_Conic
WKID (EPSG): 102003


# Re-calculate areas

In [25]:
for i in range(n_y):
    # planar area
    arcpy.management.CalculateGeometryAttributes(
        in_features=step11[i],
        geometry_property=[["Polygon_Area_Ha_Planar", "AREA"]],
        area_unit="HECTARES",
        coordinate_system=target_sr,  # NAD 1983 Albers
        coordinate_format="SAME_AS_INPUT"                  
    );

    # geodesic area
    arcpy.management.CalculateGeometryAttributes(
        in_features=step11[i],
        geometry_property=[["Polygon_Area_Ha_Geodesic", "AREA_GEODESIC"]],
        area_unit="HECTARES",
        coordinate_system=target_sr,
        coordinate_format="SAME_AS_INPUT"
    );

# Separate ag and developed layers

In [27]:
step12_Dev = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step12_NonDeep_{y}NLCD_CntyIntersect_Developed")) for y in years]

In [28]:
step12_Ag = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step12_NonDeep_{y}NLCD_CntyIntersect_Ag")) for y in years]

In [29]:
# collect polygon inclusions
for i in range(n_y):
    # export only medium intensity development (23) and high intensity development (24)
    arcpy.conversion.ExportFeatures(
        in_features = step11[i],
        out_features = step12_Dev[i],
        where_clause = "gridcode IN (23, 24)"
    )

    # export only row crops (82)
    arcpy.conversion.ExportFeatures(
        in_features = step11[i],
        out_features = step12_Ag[i],
        where_clause = "gridcode IN (82)"
    )

# Add up total converted to agriculture and developed by county

In [31]:
# column with values to group by
case_fields = ["NAME"]

In [32]:
# field to sum and statistics type
stats_fields = [["Polygon_Area_Ha_Geodesic", "SUM"]]

In [33]:
ag_county_sum_table = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step12_NonDeep_{y}NLCD_CntySum_Ag")) for y in years]

In [34]:
dev_county_sum_table = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step12_NonDeep_{y}NLCD_CntySum_Dev")) for y in years]

In [37]:
# run summary statistics
for i in range(n_y):
    arcpy.analysis.Statistics(step12_Ag[i], 
                              ag_county_sum_table[i], 
                              stats_fields, 
                              case_fields);
    arcpy.analysis.Statistics(step12_Dev[i], 
                              dev_county_sum_table[i], 
                              stats_fields, 
                              case_fields);  